In [1]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import seaborn as sb
from sympy import *

In [2]:
def U_norm_satisfier_new(U, X_ext, del_eta, eta, d_all, Umax_all):
    redo = False
    eta_hat = eta + del_eta
    X_in = np.linalg.inv(X_ext)
    U_new = eta_hat@X_in
    ##sb.heatmap(U_new-U)
    ##plt.show()
    eta_new = np.zeros((Ny,Nx))
    all_alphas = []
    crossed=[]
    problems = []
    for i in range(0, Ny):

        d = d_all[i]
        Umax = Umax_all[i]
        Umax_new = np.sum(U_new[i,:]**2)
        ##print(i, Umax_new)
        if Umax_new> Umax:
            crossed.append('crossed')
            a = np.sum(((del_eta[i, ~d])@(X_in[~d,:]))**2)

            b1 = 2*(eta[i,~d]@(X_in[~d,:]))*(del_eta[i,~d]@(X_in[~d,:]))
            b2 = 2*(eta_hat[i,d]@X_in[d,])*(del_eta[i,~d]@X_in[~d,])
            b = np.sum(b1+b2)

            c1 = (eta_hat[i,d]@X_in[d,:])**2
            c2 = ((eta[i,~d])@(X_in[~d,:]))**2
            c3 = 2*(eta_hat[i,d]@X_in[d,:])*(eta[i,~d]@X_in[~d,:])
            c = np.sum(c1+c2+c3)-Umax

            ##print(i, (b**2)-(4*a*c))
            if (b**2)-(4*a*c)>0:
                ##print(i, 'ok')
                var('x')
                alphas = solve(a*(x**2)+b*x+c, x)
                ##print(alphas)
                if alphas[0]>0 and alphas[0]<1:
                    alpha = alphas[0]
                    all_alphas.append(alpha)
                    problems.append(0)
                elif alphas[1]>0 and alphas[1]<1:
                    alpha = alphas[1]
                    all_alphas.append(alpha)
                    problems.append(0)
                else:
                    all_alphas.append('not between 0 and 1')
                    redo = True
                    ##print(i,'problem')
                    alpha =1 
                    problems.append(1)
                    ##break

                del_eta_prop = np.copy(del_eta[i,:])
                del_eta_prop[~d] = alphas[1]*del_eta[i,~d]
                eta_new[i,:] = eta[i,:] + del_eta_prop

                '''plt.plot(del_eta[i,:],'-o')
                plt.plot(del_eta_prop,'-o')
                plt.title(str(i)+', alpha = ' + str(alpha) + ', drift ' + str(np.where(d== True)[0]))
                plt.show()'''
            else:
                eta_new[i,:] = eta[i,:] + del_eta[i,:]
                ##print(i, 'problem')
                all_alphas.append('imaginary')
                redo = True
                problems.append(1)
                ##break
        else:
            crossed.append('not crossed')
            eta_new[i,:] = eta[i,:] + del_eta[i,:]
            all_alphas.append(1)
            problems.append(0)
    return(eta_new, redo, problems)

In [3]:
def robustness_plotter(new_U, U_drift, T_all, h_min, h_max,W, W_new, which_add):
    fig, axs = plt.subplots(1,3, figsize = (20,5))
    counter = 0
    mean_diff_before= []
    mean_sd_before = []
    mean_diff_after= []
    mean_sd_after = []
    ##for noise_sd in [0.01, 0.1, 1]:
    for noise_sd in [0.1]:
        U_curr_before = new_U.copy()
        U_curr_after = U_drift.copy()

        before_errors = []
        after_errors = []

        for noises in range(0, 1000):
            actual_output = np.matmul(W, np.clip(np.matmul(U_curr_before, T_all), h_min, h_max))[0]
            if which_add == 'U':
                add_noise = np.random.normal(0, noise_sd, np.shape(U_curr_before))
                add_noise_W = np.zeros_like(W)
            elif which_add == 'W':
                add_noise = np.zeros_like(U_curr_before)
                add_noise_W = np.random.normal(0, noise_sd, np.shape(W))
            elif which_add == 'UW':
                add_noise = np.random.normal(0, noise_sd, np.shape(U_curr_before))
                add_noise_W = np.random.normal(0, noise_sd, np.shape(W))
            new_output_before = np.matmul(W + add_noise_W, np.clip(np.matmul(U_curr_before + add_noise, T_all), h_min, h_max))[0]
            new_output_after = np.matmul(W_new + add_noise_W, np.clip(np.matmul(U_curr_after + add_noise, T_all), h_min, h_max))[0]
            before_errors = np.concatenate((before_errors,actual_output-new_output_before))
            after_errors = np.concatenate((after_errors,actual_output-new_output_after))

        bins=np.histogram(np.hstack((before_errors,after_errors)), bins=100)[1] #get the bin edges


        axs[counter].hist(before_errors, bins = bins, label= 'errors before drift', alpha = 0.5)

        axs[counter].hist(after_errors, bins = bins, label = 'errors after drift', alpha = 0.5)
        axs[counter].legend()
        axs[counter].set_title(str(np.shape(T_all)[1]) + ' patterns, noise sd = ' + str(noise_sd))
        axs[counter].axvline(0)
        
        counter = counter + 1
        mean_diff_before.append(np.mean(before_errors))
        mean_sd_before.append(np.std(before_errors))
        mean_diff_after.append(np.mean(after_errors))
        mean_sd_after.append(np.std(after_errors))
        
    
        
    before_errors = np.array(before_errors)
    after_errors = np.array(after_errors)
    mean_all = [mean_diff_before, mean_diff_after, mean_sd_before, mean_sd_after]
    return(mean_all, np.sum(before_errors**2)/len(before_errors), np.sum(after_errors**2)/len(after_errors))

In [4]:
def representational_drift_fixed_Unorm (U, X,W,  num_runs, h, h_up, Umax_fixed_all, Wmax_fixed, W_changes_allowed, eta_clip, eta_min=-100000, eta_max = 100000):  

    h = h_min
    h_up = h_max
    error_all =[]
    h_prime = 10**-20
    U_diff = []
    Y_diff = []
    all_problems =[]
    
    U_initial = np.copy(U)
    
    P = np.shape(X)[1]
    Nt = np.shape(U)[1]
    Nx = Nt
    Ns = np.shape(W)[1]
    Ny = Ns
    Nz = np.shape(W)[0]

    if P == 0:
        X_ex = np.random.normal(0, 1, (Nx, Nx))
    elif Nt> P:
        X_ex = np.concatenate((X, scipy.linalg.null_space(X.T)), axis = 1)
    else:
        X_ex = X
    
    eta = np.matmul(U,X_ex)
    if P != 0:
        Y = np.clip(np.matmul(U, X), h, h_up)
        Z_given_store = np.matmul(W, Y)


    Active_all = {}
    Inactive_all = {}
    Saturated_all = {}
    first_time_active = {}
    first_time_active_sat = {}
    first_time_inactive = {}
    first_time_saturated = {}
    for mu in range(0,P):
        Y_given = np.matmul(U, X)[:,mu]
        Active_all[mu] =  np.where(np.logical_and((Y_given>(h-h_prime)),Y_given< (h_up-h_prime)))[0]
        ##print('Y active', Y_given[Active_all[mu]])
        Inactive_all[mu] = np.where(Y_given<=h-h_prime)[0]
        Saturated_all[mu] = np.where(Y_given>=h_up+h_prime)[0]
        first_time_active[mu] = []
        first_time_inactive[mu] = []##np.where(eta[:,mu]==h_min)[0]
        first_time_saturated[mu] = []
        first_time_active_sat[mu] = []##np.where(eta[:,mu]==h_max)[0]
    
    num_saturated = []
    num_inactive = []
    
    ##Umax_fixed_all = np.array([5]*Ns)##ynp.sum(U**2, axis = 1)
    ##print(Umax_fixed_all)
    U_sum_all= []
    num_saturated = []
    num_inactive = []
    W_sum_all= []
    sigma_mu = 0.1 ##2##0.5##0.01
    ##h = 0 ##10**-10 ##10**-10 ##2*10**-11
    ##h_up = 5
    hprime = 10**-20
    
    ##all_errors = []
    all_Ys= []
    all_Us = []
    all_Y_nums = np.arange(0, Ns)
    W_plus = np.linalg.pinv(W)
    pi_w = np.matmul(W_plus, W)## projection matrix
    sb.heatmap(pi_w)
    plt.title('pi_w projection matrix')
    plt.show()
    
    redos = np.zeros(num_runs)
    for t in range (0, num_runs):
        ##print(t)
        
        W_plus = np.linalg.pinv(W)
        pi_w = np.matmul(W_plus, W)## projection matrix
        
        
        
        redo = True
        Active_all_prev = Active_all.copy()
        Inactive_all_prev = Inactive_all.copy()
        Saturated_all_prev = Saturated_all.copy()
        first_time_active_prev = first_time_active.copy()
        first_time_active_sat_prev = first_time_active_sat.copy()
        first_time_inactive_prev = first_time_inactive.copy()
        first_time_saturated_prev = first_time_saturated.copy()
        while redo== True:
            ##print('doing')
            Active_all = Active_all_prev.copy()
            Inactive_all = Inactive_all_prev.copy()
            Saturated_all = Saturated_all_prev.copy()
            first_time_active = first_time_active_prev.copy()
            first_time_active_sat = first_time_active_sat_prev.copy()
            first_time_inactive = first_time_inactive_prev.copy()
            first_time_saturated = first_time_saturated_prev.copy()
    
            U_norm_satisfied = False
            del_eta_con = np.zeros((Ns, Nt))
    
    
            for mu in range(0, Nt):
                if mu < P:
                    ##print(t, mu)
                    
                    Y_given = Y[:,mu]
                    Active = Active_all[mu]
                    Inactive = Inactive_all[mu]
                    Saturated = Saturated_all[mu]
                    ##print(mu, 'Active', Y_given[Active])
    
                    Y_tilda_saturated =  Y_given[Saturated]
                    Y_tilda_active =  Y_given[Active]
                    Y_tilda_inactive = Y_given[Inactive]
                    W_saturated = W[:,Saturated]
                    W_active = W[:,Active]
                    W_inactive = W[:,Inactive]
    
                    if len(Active)>0:  ## adding fails when all neurons are disengaged
                        k_w = scipy.linalg.null_space(W[:,Active])
                        no_engaged = False
                        found_first_active_nonneg = False
                        found_first_active_below = False
                        del_eta_cons_potential = np.zeros(len(Active))
                    else:
                        no_engaged = True
                        ##print(t, mu, 'no engaged')
                        ##print(Active)
                        del_eta_cons_potential=[]
    
                    if len(Active)== 1:
                        ##print(t, mu, 'one became active')
                        if np.isclose(Y_given[Active], h_min)== True:
                            del_eta_cons_potential= [-abs(np.random.normal(0,0.1))]
                        elif np.isclose(Y_given[Active], h_max)== True:
                            del_eta_cons_potential= [+abs(np.random.normal(0,0.1))]
                        else:
                            del_eta_cons_potential= [0]
                    while (found_first_active_nonneg == False or found_first_active_below == False) and no_engaged == False and len(Active)>1:
                        ##print(t, mu,'should not go')
    
                        ##print('first foudn active nonneg or found first active below')
                        den2 = np.sum(np.linalg.inv(X_ex)[mu,:]**2)
                        den3 = np.sum(k_w**2, axis = 1)
    
                        del_r = np.random.normal(0, (sigma_mu)/np.sqrt(np.sum(den2)), np.shape(k_w)[1])
                        
                        del_eta_cons_potential = np.matmul(k_w,del_r) 
                        ##print(del_eta_cons_potential)
                        if len(first_time_active[mu])!=0:
                            neur_check = []
                            for neur_check_ind in range(len( first_time_active[mu])):
                                neur_check.append(np.where(Active_all[mu]==first_time_active[mu][neur_check_ind])[0][0])
    
                            if np.all(del_eta_cons_potential[neur_check]>h):
                                found_first_active_nonneg = True
                        else:
                            found_first_active_nonneg = True
    
    
                        if len(first_time_active_sat[mu])!= 0:
                            neur_check = []
                            for neur_check_ind in range(len( first_time_active_sat[mu])):
                                neur_check.append(np.where(Active_all[mu]==first_time_active_sat[mu][neur_check_ind])[0][0])
    
                            if np.all(del_eta_cons_potential[neur_check]< 0):
                                found_first_active_below = True
    
                        else:
                            found_first_active_below = True
                            
                    del_eta_cons_potential= np.array(del_eta_cons_potential)
                    first_time_active[mu] = [] ## reset
                    first_time_active_sat[mu] = []
                    Y_changing_cons = eta[Active, mu] +del_eta_cons_potential
                    mult = 1
                    mult_sat = 1
    
                    inactivated_neuron = False
                    saturated_neuron = False
                    if len(np.where(Y_changing_cons<=h)[0])>0 or len(np.where(Y_changing_cons>=h_up)[0])>0:
                        ##print(t, mu,'solving problematic')
    
                        problematic = [Y_changing_cons<=h][0]   ## changing MN Jan 10th 2024
                        if np.sum(problematic)!=0:
                            ##print(t, mu, 'solving problematic inactive')
                            ##min_to_be_changed = np.min((abs(h-eta[Active, mu]))[tuple(problematic)]/abs(del_eta_cons_potential[problematic]))
                            min_to_be_changed = np.min((abs(h-eta[Active, mu]))[problematic]/abs(del_eta_cons_potential[problematic])) ##changing MN Jan 7th 2024
                            to_be_changed1 = np.where((abs(h-eta[Active, mu]))/abs(del_eta_cons_potential) == min_to_be_changed)[0][0]
                            mult = abs(h-eta[Active][to_be_changed1, mu])/abs(del_eta_cons_potential[to_be_changed1])
    
                        problematic_sat = [Y_changing_cons>=h_up][0] ## changing MN Jan 10th 2024
                        if np.sum(problematic_sat)!=0:
                            ##print(t, mu, 'solving problematic saturated')
                            ##min_to_be_changed_act_sat = np.min((h_up-eta[Active, mu][tuple(problematic_sat)])/abs(del_eta_cons_potential[problematic_sat]))
                            min_to_be_changed_act_sat = np.min((h_up-eta[Active, mu][problematic_sat])/abs(del_eta_cons_potential[problematic_sat]))##changing MN Jan 7th 2024
                            to_be_changed_act_sat = np.where((h_up-eta[Active, mu])/abs(del_eta_cons_potential) == min_to_be_changed_act_sat)[0][0]
                            mult_sat = (h_up-eta[Active][to_be_changed_act_sat, mu])/del_eta_cons_potential[to_be_changed_act_sat]
                        ##print('mult', mult, mult_sat)
                        
                        if mult>1:
                            print(t, mu, 'mult', Y_changing_cons,mult, min_to_be_changed, h,  eta[Active][to_be_changed1, mu], abs(h-eta[Active][to_be_changed1, mu]),abs(del_eta_cons_potential[to_be_changed1]))
                        if mult_sat>1:
                            print(t, mu,'mult_sat', Y_changing_cons,mult_sat, min_to_be_changed_act_sat, h_up, eta[Active][to_be_changed_act_sat, mu], h_up-eta[Active][to_be_changed_act_sat, mu], del_eta_cons_potential[to_be_changed_act_sat])
                        
                        if mult_sat<mult:
                            
                            saturated_neuron = True
    
                            
                            to_be_changed_act_sat_val = Active[to_be_changed_act_sat]
                            
                            Active_all[mu]= np.sort(np.setdiff1d(Active_all[mu],to_be_changed_act_sat_val))
                            ##print(mu, to_be_changed_act_sat_val, to_be_changed_act_sat)
                            Saturated_all[mu] = np.sort(np.append(Saturated_all[mu],to_be_changed_act_sat_val))
                            del_eta_con[Active,mu] = del_eta_cons_potential*mult_sat
                            
                            ##sb.heatmap(eta+del_eta_con)
                            ##print(mu, Saturated_all[mu])
    
                        else:
                            inactivated_neuron = True
                            
                            to_be_changed1_val = Active[to_be_changed1]
                            
                            Active_all[mu]= np.sort(np.setdiff1d(Active_all[mu],to_be_changed1_val))
                            Inactive_all[mu] = np.sort(np.append(Inactive_all[mu],to_be_changed1_val))
                            del_eta_con[Active,mu] = del_eta_cons_potential*mult 
                            
                            
    
    
                    else:
                        del_eta_con[Active,mu] = del_eta_cons_potential
                        '''print('modified-engaged-set')
                        plt.plot(del_eta_con[Active,mu])
                        plt.show()'''
    
    
    
                    ##print('reached inactive')
                    
                    sigma_inactive = sigma_mu/np.sqrt(np.sum(np.linalg.inv(X_ex)[mu,:]**2))
                    found_first_active_nonneg = False
                    while found_first_active_nonneg == False:
    
                        new_change = np.random.normal(0,sigma_inactive, len(Inactive))
                        if len(first_time_inactive[mu])!=0:
                            if new_change[np.where(Inactive==first_time_inactive[mu][0])[0][0]]<0:
                                found_first_active_nonneg = True
                            ##else:
                                ##print('found_first_active_nonneg')
                        else:
                            found_first_active_nonneg = True
    
                    Y_changing = eta[Inactive, mu] + new_change
    
    
                    ## check whether next line is necessary
                    ##Y_changing = eta[Inactive, mu] + np.random.normal(0,sigma_inactive, len(Inactive))
    
                    del_eta_con[Inactive,mu] = np.minimum(Y_changing, h)- eta[Inactive, mu] ## maybe problem
    
                    if len(np.where(Y_changing>=h)[0])>0:
                        ##print('first time active')
                        to_be_changed =Inactive[np.where(Y_changing>=h)[0]]
                        Active_all[mu]= np.sort(np.concatenate((Active_all[mu],to_be_changed)))
    
                        Inactive_all[mu] = np.setdiff1d(Inactive_all[mu],to_be_changed)
    
                        first_time_active[mu]= to_be_changed
                    if len(np.intersect1d(Active_all[mu], Inactive_all[mu]))!= 0:
                        print('something wrong, check, inactive, active')
    
                    first_time_inactive[mu]= [] ## rest
                    if inactivated_neuron == True:
                        first_time_inactive[mu]= [to_be_changed1_val]
    
    
                    ##print('reached saturated')
                    
                    ## saturated
                    sigma_saturated = sigma_inactive
    
                    found_first_active_below = False
                    while found_first_active_below == False:
                        new_change = np.random.normal(0,sigma_saturated, len(Saturated))
                        if len(first_time_saturated[mu])!=0:
                            if new_change[np.where(Saturated==first_time_saturated[mu][0])[0][0]]>0:
                                found_first_active_below = True
    
                        else:
                            found_first_active_below = True
    
                    Y_changing = eta[Saturated, mu] + new_change
    
    
                    del_eta_con[Saturated,mu] = np.maximum(Y_changing, h_up)- eta[Saturated, mu]
    
    
                    if len(np.where(Y_changing<=h_up)[0])>0:
                        ##print('first time active sat')
                        to_be_changed_sat =Saturated[np.where(Y_changing<h_up)[0]]
                        Active_all[mu]= np.sort(np.concatenate((Active_all[mu],to_be_changed_sat)))
    
                        Saturated_all[mu] = np.setdiff1d(Saturated_all[mu],to_be_changed_sat)
                        first_time_active_sat[mu]= to_be_changed_sat 
    
                    if len(np.intersect1d(Active_all[mu], Saturated_all[mu]))!= 0:
                        print('something wrong, check, active, saturated')
    
                    first_time_saturated[mu]=[]
                    if saturated_neuron == True:
                        ##error_plotter(np.clip(eta+del_eta_con, 0, h_up)[:,0:P])
                        first_time_saturated[mu] = [to_be_changed_act_sat_val]
                        ##print('first time saturated', first_time_saturated[mu])
    
    
    
                else:
                    ##print(t, mu)
                    ##del_eta_con[:,mu] = np.random.normal(0, (sigma_mu**2)/np.sum(np.linalg.inv(X_ex)[mu]**2) )
                    del_eta_con[:,mu] =np.random.normal(0, sigma_mu/np.sqrt(np.sum(np.linalg.inv(X_ex)[mu,:]**2)), Ns)
            
            ##sb.heatmap(del_eta_con)
            ##plt.show()
            d_all = np.full((Ny, Nx), False)
            for mu in range(len(Active_all)):
                d_all[Active_all[mu], mu]= True
            eta_new, redo, problems = U_norm_satisfier_new(U, X_ex, del_eta_con, eta, d_all,Umax_fixed_all)
            
            
            all_problems.append(problems)
            ##print(t, redo)
            if redo == True:
                redos[t] = redos[t]+1
            else:
                ##sb.heatmap(eta_new-eta)
                ##plt.show()
                eta = eta_new.copy()
                
    
            
    
            
        ##eta = eta + del_eta_con
    
        ##eta = np.clip(eta,-5, 5)
        U_prev = np.copy(U)
        
        ##eta = np.clip(eta, h-100, h_up+100)
        
        if eta_clip == True:
            eta = np.clip(eta, eta_min, eta_max)
        U = np.matmul(eta, np.linalg.inv(X_ex))
        if P!= 0:
            Y_prev = np.copy(Y)
            Y = np.clip(np.matmul(U,X), h, h_up)
    
    
        '''if np.sum(((W@np.clip(eta[:,0:P],h,h_up))-Z_given_store)**2)>0.000001:
            print(t,np.sum(((W@Y)-Z_given_store)**2),W@np.clip(eta[:,0:P],h,h_up), W@Y, Z_given_store)
            print(mu_all, un_semi)'''
            
        ##sb.heatmap(np.array(Y<h_up) * np.array(Y>h)) 
        ##plt.show()
    
        ##print(t)
        if t%100 == 0:
            
            U_sum_all.append(np.sum(U**2, axis = 1))
            U_diff.append(U-U_prev)
            if P!= 0:
                Y_diff.append(Y-Y_prev)
                all_Ys.append(Y)
            
            
            ##print(Inactive_all)
            
            num_saturated.append(np.sum(eta[:,0:P]>=h_up))
            num_inactive.append(np.sum(eta[:,0:P]<=h))
            ##all_errors.append(error_plotter(Y))
            ##W_sum_all.append(np.sum(W**2))
    
            
            all_Us.append(U)
            ##diff = W@np.clip(np.matmul(U, T_all), h, h_up)-W@np.clip(np.matmul(U_initial, T_all), h, h_up)
            ##error_all.append(np.sum(diff**2))
        if t%100 ==0:
            print(t)
    return(U, num_saturated, num_inactive, U_sum_all, W, W_sum_all, all_Us, error_all, U_diff, Y_diff, redos, all_problems)